# Step 3: Data Augmentation

用 new_features（224 筆）訓練，對 hist_features（10487 筆）推估 BASS 參數（p、q、M、constant）。

- **方法 A**：KMeans Clustering（k=10）
- **方法 B**：KNN 距離加權（K=5）

以 test set（20%）的 MAE / RMSE 決定最終使用方法。

## 設計說明：為何最終改用 KNN

開發過程中曾嘗試兩種分群方法，最終皆放棄，改用 KNN 作為唯一推估方式：

**方法 A-1：KMeans Clustering**
用 embedding + duration + AI業配判定 + best_topic_labels 做分群，以 cluster 平均值推估。問題：training data 只有 178 筆，k=20 時平均每群僅 9 筆；k=8 時 cluster 內部仍有極端值，M 值出現 1.676e+25 等不合理數字，即使加 log1p 轉換仍有分群不均問題。

**方法 A-2：Topic Median**
改用 best_topic_labels 直接分群取中位數。問題：部分 topic 在 train 只有 1～2 筆，缺乏代表性；hist_features 有些 topic 在 train 未出現，只能 fallback 到全體中位數，損失個別差異。

**最終選用 KNN（K=5）**
不需預先定義群數，直接找語意最相近的 5 筆訓練資料做 Euclidean 距離加權平均，保留個別差異且不受分群粒度影響，RMSE 表現穩定優於上述方法。

In [65]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

In [66]:
PROCESSED = Path('../data/processed')
OUTPUT    = Path('../data/output')
OUTPUT.mkdir(parents=True, exist_ok=True)

new_features  = pd.read_csv(PROCESSED / 'new_features.csv')
hist_features = pd.read_csv(PROCESSED / 'hist_features.csv')

print('new_features :', new_features.shape)
print('hist_features:', hist_features.shape)

new_features : (223, 60)
hist_features: (10373, 59)


## 1. 定義特徵欄位與資料切分

In [67]:
emb_cols    = [f'emb_{i:03d}' for i in range(32)]
target_cols = ['p_after4d', 'q_after4d', 'M_after4d', 'constant_after4d_time0_views']
out_cols    = ['p', 'q', 'M', 'constant']

# 只保留四個 target 都不為空的樣本，並做 log1p 壓縮極端值
labeled = new_features.dropna(subset=target_cols).reset_index(drop=True)
labeled[target_cols] = labeled[target_cols].apply(np.log1p)

train_df, test_df = train_test_split(labeled, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'labeled: {len(labeled)} 筆')
print(f'train  : {len(train_df)} 筆')
print(f'test   : {len(test_df)} 筆')

labeled: 223 筆
train  : 178 筆
test   : 45 筆


In [68]:
# LabelEncoder：fit 在三個資料集的聯集上，確保 hist/test 不會出現 unseen label
le = LabelEncoder()
all_labels = list(
    pd.concat([train_df['best_topic_labels'],
               test_df['best_topic_labels'],
               hist_features['best_topic_labels']])
    .dropna().unique()
) + ['unknown']
le.fit(all_labels)

num_extra = ['duration', 'AI業配判定']

def build_raw(df):
    d = df[emb_cols + num_extra].copy()
    d['best_topic_labels_enc'] = le.transform(
        df['best_topic_labels'].fillna('unknown')
    )
    return d

# 中位數和 StandardScaler 只在 train 上 fit（避免 leakage）
train_raw = build_raw(train_df)
medians   = train_raw.median()

scaler = StandardScaler()
scaler.fit(train_raw.fillna(medians))

def get_X(df):
    d = build_raw(df).fillna(medians)
    return scaler.transform(d)

X_train = get_X(train_df)
print(f'特徵維度：{X_train.shape[1]}  '
      f'（emb×32 + duration + AI業配判定 + best_topic_labels_enc）')

特徵維度：35  （emb×32 + duration + AI業配判定 + best_topic_labels_enc）


In [69]:
# 診斷：StandardScaler 後各特徵的 mean 和 std（應接近 0 和 1，異常值代表離群或 scale 問題）
feat_names = build_raw(train_df).columns.tolist()
feat_stats = pd.DataFrame({
    'mean': X_train.mean(axis=0),
    'std' : X_train.std(axis=0),
}, index=feat_names)
print('=== StandardScaler 後特徵統計 ===')
print(feat_stats.to_string())

=== StandardScaler 後特徵統計 ===
                               mean  std
emb_000               -9.480556e-17  1.0
emb_001                1.072800e-16  1.0
emb_002               -1.896111e-16  1.0
emb_003               -4.490790e-16  1.0
emb_004                9.979533e-17  1.0
emb_005                4.241301e-17  1.0
emb_006               -4.490790e-17  1.0
emb_007               -7.584445e-16  1.0
emb_008               -2.494883e-18  1.0
emb_009               -1.496930e-17  1.0
emb_010               -1.447032e-16  1.0
emb_011                0.000000e+00  1.0
emb_012                2.494883e-17  1.0
emb_013               -1.047851e-16  1.0
emb_014                1.779164e-16  1.0
emb_015                1.496930e-17  1.0
emb_016               -2.569730e-16  1.0
emb_017                5.488743e-17  1.0
emb_018               -7.285059e-16  1.0
emb_019               -3.293246e-16  1.0
emb_020               -1.995907e-17  1.0
emb_021                1.846214e-16  1.0
emb_022               -1.197

## 2. 方法：KNN 距離加權（K=5）——唯一採用方法

In [71]:
K_NEIGHBORS = 5

knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric='euclidean')
knn.fit(X_train)

def predict_knn(df):
    distances, idx = knn.kneighbors(get_X(df))
    weights = 1.0 / (distances + 1e-8)
    weights = weights / weights.sum(axis=1, keepdims=True)
    preds = {}
    for col in target_cols:
        neighbor_vals = train_df[col].values[idx]  # (n, K)
        preds[col]    = (weights * neighbor_vals).sum(axis=1)
    return pd.DataFrame(preds)

test_pred_b = predict_knn(test_df)
print('方法 B 預測範例（前 3 筆）:')
print(test_pred_b.head(3))

方法 B 預測範例（前 3 筆）:
      p_after4d  q_after4d  M_after4d  constant_after4d_time0_views
0  8.495877e-02   0.057409  19.791718                     10.470564
1  6.239626e-02   0.136263  10.438343                      9.957237
2  6.159539e-07   0.024815  26.859802                     11.491891


## 4. 交叉驗證（Test Set 評估）

In [72]:
# target 已做 log1p，預測值也在 log scale，MAE/RMSE 直接在 log scale 下比較
rows = []
for col, label in zip(target_cols, out_cols):
    y = test_df[col].values
    rows.append({
        'target': label,
        'MAE'   : mean_absolute_error(y, test_pred_b[col]),
        'RMSE'  : np.sqrt(mean_squared_error(y, test_pred_b[col])),
    })

eval_df = pd.DataFrame(rows)
print(eval_df.to_string(index=False))
print(f'\nKNN RMSE 總和（log scale）：{eval_df["RMSE"].sum():.4f}')

  target      MAE      RMSE
       p 0.064437  0.084761
       q 0.126534  0.187685
       M 7.251263 26.217918
constant 1.519371  2.358785

KNN RMSE 總和（log scale）：28.8491


## 5. 選擇最佳方法並輸出 augmented_hist.csv

In [73]:
best_method  = 'KNN'
hist_pred_df = predict_knn(hist_features)

hist_pred_df = hist_pred_df.rename(columns=dict(zip(target_cols, out_cols)))

# log scale → 原始尺度
for col in out_cols:
    hist_pred_df[col] = np.expm1(hist_pred_df[col])

hist_pred_df.insert(0, 'reels_shortcode', hist_features['reels_shortcode'].values)
hist_pred_df['method_used'] = best_method

out_path = OUTPUT / 'augmented_hist.csv'
hist_pred_df.to_csv(out_path, index=False)
print(f'選用方法：{best_method}')
print(f'已儲存：{out_path}')
print(hist_pred_df.shape)
print(hist_pred_df.head())

選用方法：KNN
已儲存：../data/output/augmented_hist.csv
(10373, 6)
  reels_shortcode         p         q              M      constant method_used
0     DPeT4BHExo-  0.162187  0.026166   68803.130050  25855.820699         KNN
1     DPZEm_oExnd  0.087516  0.012976   42449.616128  22533.613050         KNN
2     DLvRZSlTBzn  0.014141  0.078649  557076.289925  16552.773160         KNN
3     DIvNtToTBL-  0.123744  0.009191  989863.165004  44753.210573         KNN
4     Ciksu1-vDBo  0.140977  0.103586   57409.677393  30904.614276         KNN
